In [112]:
import torch
import timm
import torch.nn as nn
from models.resnet18_cifar import resnet18
# create arguments
import argparse


parser = argparse.ArgumentParser(description='PyTorch CIFAR10 Training')
parser.add_argument('--model', default='resnet18', type=str, help='model name (default: resnet18)')
parser.add_argument('--dataset', default='seq-cifar10', type=str, help='dataset name (default: seq-cifar10)')
parser.add_argument('--batch-size', default=128, type=int, help='batch size (default: 128)')
parser.add_argument('--epochs', default=200, type=int, help='number of total epochs to run')
parser.add_argument('--lr', default=0.1, type=float, help='initial learning rate')

args = parser.parse_args()

args.dataset = 'seq-cifar10'
args.model = 'resnet18'
args.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [58]:
args

Namespace(model='resnet18', dataset='seq-cifar10', batch_size=128, epochs=200, lr=0.1, device=device(type='cpu'))

# VPT 

In [ ]:
# ✅ VPT-Shallow: Learnable Prompt 벡터 추가
PROMPT_DIM = 32  # 프롬프트 벡터 크기
NUM_PROMPTS = 5  # 추가할 프롬프트 개수

class VPT_ViT(nn.Module):
    def __init__(self, model, num_prompts=NUM_PROMPTS, prompt_dim=PROMPT_DIM):
        super(VPT_ViT, self).__init__()
        self.feat = model
        self.num_classes = model.num_classes
        self.num_prompts = num_prompts
        self.prompt_dim = prompt_dim

        # 📌 학습할 프롬프트 벡터 정의
        self.prompt_embeddings = nn.Parameter(
            torch.randn(1, num_prompts, self.feat.embed_dim) * 0.02  # (1, P, D)
        )

        # ✅ Positional Embedding 확장 (197 → 197+num_prompts)
        old_pos_embed = self.feat.pos_embed  # 기존 positional embedding 저장
        new_pos_embed = nn.Parameter(
            torch.cat([old_pos_embed[:, :1, :],  # CLS token 유지
                       torch.randn(1, num_prompts, model.embed_dim) * 0.02,  # VPT prompts 추가
                       old_pos_embed[:, 1:, :]], dim=1)  # 기존 patch embeddings 유지
        )
        self.feat.pos_embed = new_pos_embed  # 새로운 positional embedding 적용

        # ✅ Classifier head (CIFAR-100)
        self.head = nn.Linear(model.embed_dim, self.num_classes)

        self.feat.requires_grad_(False)
        self.head.requires_grad_(True)
        self.prompt_embeddings.requires_grad_(True)  # 프롬프트 벡터 학습 가능하도록 설정

    def forward(self, x):
        B = x.shape[0]  # 배치 크기
        x = self.feat.patch_embed(x)  # 패치 임베딩 적용

        # 📌 VPT: Learnable Prompt 추가
        prompt_embed = self.prompt_embeddings.expand(B, -1, -1)  # (B, P, D)
        x = torch.cat([prompt_embed, x], dim=1)  # (B, P+197, D)

        # ✅ Positional Embedding 적용
        x = self.feat.pos_drop(x + self.feat.pos_embed[:, :x.shape[1], :])  # 크기 맞추기
        x = self.feat.blocks(x)  # Transformer 블록 통과
        x = self.feat.norm(x[:, 0])  # CLS 토큰 사용
        x = self.head(x)  # 분류기 적용
        return x

In [59]:
resnet18_model = resnet18(dataset=args.dataset).to(args.device)

In [60]:
# Calculate FLOPS
def print_flops(model, input_tensor):
    flops = 0
    def count_flops(module, input, output):
        nonlocal flops
        if isinstance(module, torch.nn.Conv2d):
            flops += input[0].numel() * module.weight.numel() / module.groups
        elif isinstance(module, torch.nn.Linear):
            flops += input[0].numel() * module.weight.numel()
    hooks = []
    for layer in model.modules():
        hooks.append(layer.register_forward_hook(count_flops))
    with torch.no_grad():
        model(input_tensor)
    for hook in hooks:
        hook.remove()
    return flops

In [61]:
resnet18_flops = print_flops(resnet18_model, torch.randn(1, 3, 32, 32).to(args.device))
resnet18_flops = resnet18_flops / 1e9
print(f"FLOPS: {resnet18_flops:.2f} GFLOPS")

FLOPS: 148.72 GFLOPS


In [62]:
resnet18_flops = print_flops(resnet18_model, torch.randn(1, 3, 224, 224).to(args.device))
resnet18_flops = resnet18_flops / 1e9
print(f"FLOPS: {resnet18_flops:.2f} GFLOPS")

FLOPS: 7287.21 GFLOPS


In [81]:
# Calculate Parameters, Memory, Model Size and Latency
def print_params(model):
    total_params = sum(p.numel() for p in model.parameters())
    total_memory = total_params * 4 / (1024 ** 2)  # Convert to MB
    model_size = total_params * 4 / (1024 ** 2)  # Convert to MB
    return total_params, total_memory, model_size


# cuda is not available
def print_cpu_latency(model, input_shape=(1, 3, 32, 32), runs=100):
    import time
    input_tensor = torch.randn(input_shape).to(args.device)
    # Average latency over 100 runs
    avg_latency = 0
    for i in range(runs):
        start_time = time.time()
        model(input_tensor)
        end_time = time.time()
        avg_latency += (end_time - start_time) * 1000  # Convert to ms

    return avg_latency / runs

In [82]:
params, memory, model_size = print_params(resnet18_model)
print(f"Parameters: {params / 1e6:.2f} M")
print(f"Memory: {memory:.2f} MB")
print(f"Model Size: {model_size:.2f} MB")
print(f"Latency: {print_cpu_latency(resnet18_model):.2f} ms")
print(f"Latency: {print_cpu_latency(resnet18_model, input_shape=(1, 3, 224, 224)):.2f} ms")

Parameters: 11.17 M
Memory: 42.63 MB
Model Size: 42.63 MB
Latency: 8.98 ms
Latency: 190.85 ms


In [88]:
params, memory, model_size = print_params(resnet18_model)
params

11173962

In [91]:
# 2197102 / (8788370 + 2197102)
sparse_params = 2197102
print(f"Parameters: {sparse_params / 1e6:.2f} M")

Parameters: 2.20 M


In [77]:
# print current torch backend if it's mps
print(f"Current torch backend: {torch.backends.mps.is_available()}")

Current torch backend: True


In [83]:
vit_model = timm.create_model('vit_tiny_patch16_224', pretrained=False)

In [84]:
vit_flops = print_flops(vit_model, input_tensor=torch.randn(1, 3, 224, 224).to(args.device))
vit_flops = vit_flops / 1e9
print(f"ViT FLOPS: {vit_flops:.2f} GFLOPS")

ViT FLOPS: 423.80 GFLOPS


In [85]:
params, memory, model_size = print_params(vit_model)
print(f"Parameters: {params / 1e6:.2f} M")
print(f"Memory: {memory:.2f} MB")
print(f"Model Size: {model_size:.2f} MB")
print(f"Latency: {print_cpu_latency(vit_model, input_shape=(1, 3, 224, 224)):.2f} ms")

Parameters: 5.72 M
Memory: 21.81 MB
Model Size: 21.81 MB
Latency: 17.89 ms


In [142]:
model = timm.create_model('vit_tiny_patch16_224', pretrained=False)
vpt_model = VPT_ViT(model=model)

In [143]:
vpt_flops = print_flops(vpt_model, input_tensor=torch.randn(1, 3, 224, 224).to(args.device))
vpt_flops = vpt_flops / 1e9
print(f"ViT FLOPS: {vpt_flops:.2f} GFLOPS")

ViT FLOPS: 431.96 GFLOPS


In [144]:
params, memory, model_size = print_params(vpt_model)
print(f"Parameters: {params / 1e6:.2f} M")
print(f"Memory: {memory:.2f} MB")
print(f"Model Size: {model_size:.2f} MB")
print(f"Latency: {print_cpu_latency(vpt_model, input_shape=(1, 3, 224, 224)):.2f} ms")

Parameters: 5.91 M
Memory: 22.55 MB
Model Size: 22.55 MB
Latency: 15.39 ms


In [137]:
# Calculate Parameters, Memory, Model Size and Latency using torchsummary and torchinfo
from torchinfo import summary as info_summary

In [148]:
# profile the model forward and backward pass using torchinfo
def profile_model(model, input_shape=(1, 3, 224, 224)):
    model.eval()
    input_tensor = torch.randn(input_shape).to(args.device)
    model_info = info_summary(model, input_data=input_tensor, verbose=0)
    return model_info

In [110]:
model_info = profile_model(resnet18_model, input_shape=(16, 3, 32, 32))
print(f"Input shape: {model_info.input_size}")
print(f"Parameters: {model_info.total_params / 1e6:.2f} M")
# print(f"Latency: {print_cpu_latency(vit_model, input_shape=(1, 3, 224, 224)):.2f} ms")

Input shape: torch.Size([16, 3, 32, 32])
Parameters: 11.17 M


In [111]:
model_info

Layer (type:depth-idx)                        Output Shape              Param #
ResNet                                        [16, 10]                  --
├─Sequential: 1-1                             --                        --
│    └─Conv2d: 2-1                            [16, 64, 32, 32]          1,728
│    └─BatchNorm2d: 2-2                       [16, 64, 32, 32]          128
│    └─Sequential: 2-3                        [16, 64, 32, 32]          --
│    │    └─BasicBlock: 3-1                   [16, 64, 32, 32]          73,984
│    │    └─BasicBlock: 3-2                   [16, 64, 32, 32]          73,984
│    └─Sequential: 2-4                        [16, 128, 16, 16]         --
│    │    └─BasicBlock: 3-3                   [16, 128, 16, 16]         230,144
│    │    └─BasicBlock: 3-4                   [16, 128, 16, 16]         295,424
│    └─Sequential: 2-5                        [16, 256, 8, 8]           --
│    │    └─BasicBlock: 3-5                   [16, 256, 8, 8]           9

In [ ]:
model_info = profile_model(vit_model, input_shape=(16, 3, 224, 224))
print(f"Input shape: {model_info.input_size}")
print(f"Parameters: {model_info.total_params / 1e6:.2f} M")
# print(f"Latency: {print_cpu_latency(vit_model, input_shape=(1, 3, 224, 224)):.2f} ms")

In [106]:
model_info

Layer (type:depth-idx)                   Output Shape              Param #
VisionTransformer                        [16, 1000]                38,016
├─PatchEmbed: 1-1                        [16, 196, 192]            --
│    └─Conv2d: 2-1                       [16, 192, 14, 14]         147,648
│    └─Identity: 2-2                     [16, 196, 192]            --
├─Dropout: 1-2                           [16, 197, 192]            --
├─Identity: 1-3                          [16, 197, 192]            --
├─Identity: 1-4                          [16, 197, 192]            --
├─Sequential: 1-5                        [16, 197, 192]            --
│    └─Block: 2-3                        [16, 197, 192]            --
│    │    └─LayerNorm: 3-1               [16, 197, 192]            384
│    │    └─Attention: 3-2               [16, 197, 192]            148,224
│    │    └─Identity: 3-3                [16, 197, 192]            --
│    │    └─Identity: 3-4                [16, 197, 192]            --


In [149]:
model_info = profile_model(vpt_model, input_shape=(16, 3, 224, 224))
print(f"Input shape: {model_info.input_size}")
print(f"Parameters: {model_info.total_params / 1e6:.2f} M")
# print(f"Latency: {print_cpu_latency(vpt_model, input_shape=(1, 3, 224, 224)):.2f} ms")

Input shape: torch.Size([16, 3, 224, 224])
Parameters: 5.91 M


In [150]:
model_info

Layer (type:depth-idx)                        Output Shape              Param #
VPT_ViT                                       [16, 1000]                960
├─VisionTransformer: 1-1                      --                        231,976
│    └─PatchEmbed: 2-1                        [16, 196, 192]            --
│    │    └─Conv2d: 3-1                       [16, 192, 14, 14]         (147,648)
│    │    └─Identity: 3-2                     [16, 196, 192]            --
│    └─Dropout: 2-2                           [16, 201, 192]            --
│    └─Sequential: 2-3                        [16, 201, 192]            --
│    │    └─Block: 3-3                        [16, 201, 192]            (444,864)
│    │    └─Block: 3-4                        [16, 201, 192]            (444,864)
│    │    └─Block: 3-5                        [16, 201, 192]            (444,864)
│    │    └─Block: 3-6                        [16, 201, 192]            (444,864)
│    │    └─Block: 3-7                        [16, 201

In [147]:
[n for n, p in vpt_model.named_parameters() if p.requires_grad == True]

['head.weight', 'head.bias']

In [156]:
import torch
# from torchvision.models import vit_b_16
import torch.nn as nn
from torch.profiler import profile, ProfilerActivity

# # 모델 생성 및 수정
# vit = vit_b_16(weights=None)
# vit.head = nn.Linear(vit.head.in_features, 100)

model = timm.create_model('vit_tiny_patch16_224', num_classes=100, pretrained=False)
model = VPT_ViT(model)#.cuda()
model.train()

input_tensor = torch.randn(1, 3, 224, 224)#.cuda()  # 💡 배치 크기 줄이기
target = torch.randint(0, 100, (1,))#.cuda()
criterion = nn.CrossEntropyLoss()

# 프로파일링
with profile(
    activities=[ProfilerActivity.CPU], #, ProfilerActivity.CUDA],
    profile_memory=True,
    record_shapes=True
) as prof:
    output = model(input_tensor)
    loss = criterion(output, target)
    loss.backward()

print(prof.key_averages().table(sort_by="cuda_memory_usage", row_limit=15))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           aten::conv2d         0.04%       3.667us         2.32%     194.757us     194.757us     147.00 Kb           0 b             1  
                                      aten::convolution         0.06%       5.125us         2.28%     191.090us     191.090us     147.00 Kb           0 b             1  
                                     aten::_convolution         0.17%      14.250us         2.22%     185.965us     185.965us     147.00 Kb           

In [157]:
import torch
from torch.profiler import profile, ProfilerActivity, tensorboard_trace_handler


with profile(
    activities=[ProfilerActivity.CPU], #, ProfilerActivity.CUDA],
    schedule=torch.profiler.schedule(wait=1, warmup=1, active=2),  # 스케줄 설정 (짧게 해도 됨)
    on_trace_ready=tensorboard_trace_handler('./logdir'),  # 💡 TensorBoard 로그 경로
    record_shapes=True,
    profile_memory=True,
    with_stack=True  # 스택 트레이스 추적
) as prof:
    for step in range(4):  # 최소 4번 step 돌려야 전체 schedule이 돈다
        output = model(input_tensor)
        loss = criterion(output, target)
        loss.backward()
        prof.step()  # 💡 매 스텝마다 호출 필요

# Analysis of the memory usage on backward pass w/ prompt and w/o prompt

In [161]:
import numpy as np


num_blocks = len(vit_model.blocks)
output_shape = [16, 201, 192]

In [164]:
prompt_start_idx = 0
memory_usage = 0
mem_fp32 = 4  # 4 bytes for float32
for i in range(num_blocks):
    mem_blcok = sum([j * mem_fp32 for j in output_shape])
    print(f"Block {i}: {mem_blcok / 1024:.2f} MB")

Block 0: 1.60 MB
Block 1: 1.60 MB
Block 2: 1.60 MB
Block 3: 1.60 MB
Block 4: 1.60 MB
Block 5: 1.60 MB
Block 6: 1.60 MB
Block 7: 1.60 MB
Block 8: 1.60 MB
Block 9: 1.60 MB
Block 10: 1.60 MB
Block 11: 1.60 MB


In [194]:
each_block = 16 * 201 * 192
total = each_block * 12
fwd_bwd_size_MB = (2 * total * 4) / (1024 ** 2)
fwd_bwd_size_MB

56.53125

In [195]:
each_block = 16 * 201 * 192
used_blocks_ratio = 0.5
num_used_blocks = int(num_blocks * used_blocks_ratio)
total = num_used_blocks * each_block
bwd_size_mb = total * mem_fp32 / (1024 ** 2)
print(f"Total memory usage: {bwd_size_mb:.2f} MB")

Total memory usage: 14.13 MB


In [178]:
# How much memory is used for the batch size of 16 images (3, x 224 x 224)
batch_size = 16
input_shape = (batch_size, 3, 224, 224)
input_tensor = torch.randn(input_shape)
# Calculate the memory usage of the input tensor
input_memory = np.prod(input_shape) * mem_fp32 / 1024
print(f"Input memory usage: {input_memory:.2f} MB")

Input memory usage: 9408.00 MB


In [165]:
import torch

def print_tensor_memory(tensor, name="Tensor"):
    numel = tensor.numel()
    size_mb = numel * tensor.element_size() / (1024**2)
    print(f"{name}: {tensor.shape}, {size_mb:.2f} MB")

In [170]:
from torch.profiler import profile, ProfilerActivity

input = torch.randn(1, 3, 224, 224).to(args.device)
with profile(activities=[ProfilerActivity.CPU], record_shapes=True, profile_memory=True) as prof:
    output = model(input)
    loss = criterion(output, target)
    loss.backward()

print(prof.key_averages().table(sort_by="self_cuda_memory_usage"))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           aten::conv2d         0.02%       5.583us         2.55%     780.919us     780.919us     147.00 Kb           0 b             1  
                                      aten::convolution         0.04%      11.124us         2.53%     775.336us     775.336us     147.00 Kb           0 b             1  
                                     aten::_convolution         0.31%      93.669us         2.49%     764.212us     764.212us     147.00 Kb           

In [ ]:
# sort by cpu_memory_usage
print(prof.key_averages().table(sort_by="cpu_memory_usage", row_limit=15))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                           aten::linear         0.87%     267.127us        66.47%      20.381ms     415.942us      15.90 Mb           0 b            49  
                                            aten::addmm        59.80%      18.334ms        64.47%      19.766ms     403.381us      15.90 Mb      15.90 Mb            49  
                                             aten::gelu         8.20%       2.514ms         8.20%       2.514ms     209.522us       7.07 Mb       7.07

In [198]:
import torch
import torch.nn as nn
from ptflops import get_model_complexity_info

# ✅ 예시 모델 정의 (예: ViT or custom prompt model)
from torchvision.models import resnet18

model = resnet18()  # or your Vision Transformer, Prompt-based model

# 모델을 evaluation 모드로 설정
model.eval()

# 입력 이미지 사이즈 (예: 3x224x224)
input_res = (3, 32, 32)

# with torch.cuda.device(0):
macs, params = get_model_complexity_info(model, input_res, as_strings=True,
                                            print_per_layer_stat=True, verbose=True)
print(f"FLOPs (MACs x2): {macs}")
print(f"Params: {params}")


ResNet(
  11.69 M, 100.000% Params, 37.69 MMac, 99.830% MACs, 
  (conv1): Conv2d(9.41 k, 0.080% Params, 2.41 MMac, 6.379% MACs, 3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(128, 0.001% Params, 32.77 KMac, 0.087% MACs, 64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(0, 0.000% Params, 16.38 KMac, 0.043% MACs, inplace=True)
  (maxpool): MaxPool2d(0, 0.000% Params, 16.38 KMac, 0.043% MACs, kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    147.97 k, 1.266% Params, 9.49 MMac, 25.127% MACs, 
    (0): BasicBlock(
      73.98 k, 0.633% Params, 4.74 MMac, 12.563% MACs, 
      (conv1): Conv2d(36.86 k, 0.315% Params, 2.36 MMac, 6.249% MACs, 64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(128, 0.001% Params, 8.19 KMac, 0.022% MACs, 64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(0, 0.000% Pa

In [199]:
37.75 * 0.2

7.550000000000001